In [4]:
%pip install -q lyricsgenius python-dotenv pandas

Note: you may need to restart the kernel to use updated packages.


In [5]:
import os
from pathlib import Path

import pandas as pd
import lyricsgenius
from dotenv import load_dotenv

load_dotenv(Path('..') / '.env')

GENIUS_ACCESS_TOKEN = os.getenv('GENIUS_ACCESS_TOKEN')

if not GENIUS_ACCESS_TOKEN:
    raise ValueError('Missing GENIUS_ACCESS_TOKEN in .env')

## 1) Initialise Genius client

In [6]:
genius = lyricsgenius.Genius(
    GENIUS_ACCESS_TOKEN,
    timeout=15,
    retries=3,
    remove_section_headers=True,
    skip_non_songs=True
)
genius.verbose = False
print('Genius client ready.')

Genius client ready.


---
## Build per-region top songs from Spotify Charts

Reads the downloaded Spotify Charts CSVs and keeps only each region's top-ranked song.
No language detection is applied at this stage.

Output columns: `rank`, `artist`, `title`, `region`, `spotify_uri`, `streams`, `weeks_on_chart`, `peak_rank`

In [ ]:
from pathlib import Path
import pandas as pd

print('Top-song extraction helpers loaded.')

Language detection helpers loaded.


In [ ]:
DATE_PATH = '2026/03/05'
RAW_DIR = Path('..') / 'data' / 'raw' / DATE_PATH

FILE_REGION_MAP = {
    'regional-us-weekly-2026-03-05.csv': 'USA',
    'regional-co-weekly-2026-03-05.csv': 'Colombia',
    'regional-tw-weekly-2026-03-05.csv': 'Taiwan',
    'regional-global-weekly-2026-03-05.csv': 'Global',
}

top_rows = []
for filename, region in FILE_REGION_MAP.items():
    df = pd.read_csv(RAW_DIR / filename)
    df = df.rename(columns={'artist_names': 'artist', 'track_name': 'title'})
    df['rank'] = pd.to_numeric(df['rank'], errors='coerce')
    df = df.dropna(subset=['rank']).sort_values('rank')

    top = df.iloc[0].copy()
    top_rows.append({
        'rank': int(top['rank']),
        'artist': top['artist'],
        'title': top['title'],
        'region': region,
        'spotify_uri': str(top['uri']).replace('spotify:track:', ''),
        'streams': top.get('streams'),
        'weeks_on_chart': top.get('weeks_on_chart'),
        'peak_rank': top.get('peak_rank'),
    })

titles_df = pd.DataFrame(top_rows).sort_values('region').reset_index(drop=True)

print(f'Total rows (one per region): {len(titles_df)}')
titles_df

Total rows: 400
region    language
Colombia  latin       100
Global    latin       100
Taiwan    latin        44
          zh           56
USA       latin       100


In [ ]:
titles_df

Saved 400 titles -> ../data/processed/titles/2026/03/05/titles.csv


,rank,artist,title,region,language,spotify_uri,streams,weeks_on_chart,peak_rank
0,1,"Mr Plata, El Americano 4KT",Las Muñequitas,Colombia,latin,4nJJCRYru4QQakCiUA155f,1990353,9,1
1,2,"ARIA VEGA, Ryan Castro",CHÉVERE (premium_remix),Colombia,latin,3CBEVPwR3kUXDoTx1lqFUQ,1793988,3,2
2,3,"Ryan Castro, Kapo, Gangsta",LA VILLA,Colombia,latin,2ZyrAym0sRLwt4PhGotHuI,1662672,18,1
3,4,Kris R.,GANAS,Colombia,latin,4KE9Ne3hgh18B3Th4xcylg,1555655,7,2
4,5,"W Sound, Beéle, Ovy On The Drums",La Plena - W Sound 05,Colombia,latin,6iOndD4OFo7GkaDypWQIou,1187848,54,1
5,6,Beéle,no tiene sentido,Colombia,latin,1HEwEN64NjgTaHmo7LfkX8,1147408,42,2
6,7,"J Balvin, Ryan Castro, DJ Snake",Tonto,Colombia,latin,7mU1fei7P9h4mpjP2Otdw5,1138980,1,7
7,8,Beéle,quédate,Colombia,latin,6VfL3MEuYeJbDlD8m011HR,1124708,42,2
8,9,Grupo Firme,El Beneficio De La Duda,Colombia,latin,5yXt80BNZGbmHFd0NHZHNn,1105825,54,3
9,10,"Yeison Jimenez, Luis Alfonso",Destino Final,Colombia,latin,2E4TYekUduml1DWIqQWNcj,1099832,13,1


---
## Genius lyrics fetch → chunked writes to `lyrics.csv`

Fetches lyrics for each region's top song and writes incrementally to:
`data/processed/titles/2026/03/05/lyrics.csv`

- Appends every chunk (safe to resume)
- Skips songs already present in output
- Avoids losing all progress if a run fails

In [ ]:
import time
from pathlib import Path
import pandas as pd

DATE_PATH = '2026/03/05'
LYRICS_OUT = Path('..') / 'data' / 'processed' / 'titles' / DATE_PATH / 'lyrics.csv'

CHUNK_SIZE = 10       # write to disk every N fetched songs
SLEEP_BETWEEN = 0.4   # seconds between Genius API calls

LYRICS_OUT.parent.mkdir(parents=True, exist_ok=True)

required_cols = [
    'rank', 'artist', 'title', 'region', 'spotify_uri',
    'streams', 'weeks_on_chart', 'peak_rank', 'lyrics'
]

Loaded 400 titles
Starting fresh fetch


In [ ]:
def fetch_lyrics_genius(client, title: str, artist: str) -> str:
    """Return lyrics string or empty string on failure."""
    try:
        hit = client.search_song(title=title, artist=artist)
        if hit and hit.lyrics:
            return hit.lyrics.strip()
    except Exception as e:
        print(f'  [warn] {title!r} by {artist!r}: {e}')
    return ''

def append_chunk(rows_chunk: list[dict], out_path: Path) -> None:
    """Append a list of row dicts to the CSV, writing the header only if the file doesn't exist yet."""
    if not rows_chunk:
        return
    chunk_df = pd.DataFrame(rows_chunk)[required_cols]
    write_header = not out_path.exists()
    chunk_df.to_csv(out_path, mode='a', header=write_header, index=False)

# ── Determine which songs still need lyrics ──────────────────────────────────
if LYRICS_OUT.exists():
    existing = pd.read_csv(LYRICS_OUT)
    fetched_uris = set(existing['spotify_uri'].astype(str))
    print(f'Found {len(fetched_uris)} songs already in {LYRICS_OUT.name}')
else:
    fetched_uris = set()
    print(f'{LYRICS_OUT.name} not found — starting fresh')

pending = titles_df[~titles_df['spotify_uri'].astype(str).isin(fetched_uris)].reset_index(drop=True)
total = len(pending)

if total == 0:
    print('All songs already fetched — nothing to do.')
else:
    print(f'Songs to fetch: {total}')
    rows_buffer = []
    for i, row in pending.iterrows():
        lyrics = fetch_lyrics_genius(genius, row['title'], row['artist'])
        rows_buffer.append({**row.to_dict(), 'lyrics': lyrics})

        status = 'ok' if lyrics else 'missing'
        print(f'[{i+1}/{total}] {row["region"]:12s} | {status} | {row["title"][:50]}')

        if len(rows_buffer) >= CHUNK_SIZE:
            append_chunk(rows_buffer, LYRICS_OUT)
            print(f'  \u2192 flushed {len(rows_buffer)} rows')
            rows_buffer = []

        time.sleep(SLEEP_BETWEEN)

    # flush remaining rows
    if rows_buffer:
        append_chunk(rows_buffer, LYRICS_OUT)
        print(f'  \u2192 flushed final {len(rows_buffer)} rows')

    print(f'Done. Output: {LYRICS_OUT}')

[1/400] Colombia     | ok | Las Muñequitas
[2/400] Colombia     | ok | CHÉVERE (premium_remix)
[3/400] Colombia     | ok | LA VILLA
[4/400] Colombia     | ok | GANAS
[5/400] Colombia     | ok | La Plena - W Sound 05
[6/400] Colombia     | ok | no tiene sentido
[7/400] Colombia     | ok | Tonto
[8/400] Colombia     | ok | quédate
[9/400] Colombia     | ok | El Beneficio De La Duda
[10/400] Colombia     | ok | Destino Final
[11/400] Colombia     | ok | NUEVA YORK
[12/400] Colombia     | ok | YOGURCITO
[13/400] Colombia     | ok | mi refe
[14/400] Colombia     | ok | BAILE INoLVIDABLE
[15/400] Colombia     | ok | FOREVER TU GANTEL
[16/400] Colombia     | ok | AMISTA
[17/400] Colombia     | ok | YOGURCITO REMIX (feat. Kris R., ROA)
[18/400] Colombia     | ok | Por Qué la Envidia
[19/400] Colombia     | ok | COQUETA
[20/400] Colombia     | ok | "EMHDM"
[21/400] Colombia     | ok | SANKA
[22/400] Colombia     | ok | DtMF
[23/400] Colombia     | ok | DÓNDE
[24/400] Colombia     | ok | LA CANC

In [ ]:
lyrics_df = pd.read_csv(LYRICS_OUT)

print(f'Saved rows: {len(lyrics_df)}')
print(f'Output path: {LYRICS_OUT}')

coverage = (lyrics_df
    .assign(has_lyrics=lyrics_df['lyrics'].fillna('').str.strip().ne(''))
    .groupby('region')['has_lyrics']
    .agg(['sum', 'count'])
    .rename(columns={'sum': 'with_lyrics', 'count': 'total'})
)
coverage['pct'] = (coverage['with_lyrics'] / coverage['total'] * 100).round(1)
print('\nLyrics coverage by region:')
print(coverage.to_string())

lyrics_df.head(10)

Raw     → ../data/raw/2026/03/05/lyrics_raw.csv
Modeling → ../data/processed/2026/03/05/lyrics_for_modeling.csv

Lyrics coverage by region:
          with_lyrics  total   pct
region                            
Colombia           93    100  93.0
Global             99    100  99.0
Taiwan             75    100  75.0
USA                98    100  98.0
